# SMART AND common split — FINAL corrected analytical sample

This notebook is the corrected version for the manuscript benchmark. It hard-checks that the benchmark uses the final analytical sample (`n = 3,767`) and a common held-out test set of `n = 942`. If it accidentally uses the raw `n = 4,024` sample, it stops with an error.

**Run with Kernel → Restart & Run All** to avoid stale outputs.

# SMART AND common-split benchmark — FINAL clean version

This notebook runs a **strategy-level benchmark** for victim–perpetrator overlap detection under a **single common held-out test split**.

It trains, on the same common split:
1. a role-specific victimization classifier,
2. a role-specific perpetration classifier,
3. a direct overlap classifier,

and compares the direct overlap classifier with a derived **Smart AND** strategy:

```text
Smart AND = victimization prediction AND perpetration prediction
```

Important: this is a **strategy-level benchmark**, not the direct combination of the separately optimized final saved classifiers.


In [1]:
# ============================================================
# 0. CONFIG
# ============================================================

from pathlib import Path
import os
import json
import random
import warnings
import numpy as np
import pandas as pd

RANDOM_STATE = 42
TEST_SIZE = 0.25
PCA_VARIANCE_THRESHOLD = 0.95

# Hard checks to avoid accidentally using the full raw n=4,024 sample
EXPECTED_ANALYTIC_N = 3767
EXPECTED_TEST_N = 942

# Operating thresholds
VICTIM_THRESHOLD_DEFAULT = 0.50
PERP_THRESHOLD_DEFAULT = 0.50
OVERLAP_DIRECT_THRESHOLD = 0.50

# Final-style overlap logistic weighting
OVERLAP_POS_WEIGHT_MULTIPLIER = 1.5

# Victimization tree selection criterion under common split
VICTIM_MIN_RECALL_FOR_TREE_SELECTION = 0.85

# Perpetration DNN settings
USE_TENSORFLOW_DNN = True
NN_EPOCHS = 200
NN_BATCH_SIZE = 128
NN_PATIENCE = 20
NN_VALIDATION_SPLIT = 0.15
PERP_POS_WEIGHT_MULTIPLIER = 1.0

# Smart AND threshold grid
GRID_STEP = 0.01
GRID_MIN = 0.01
GRID_MAX = 0.99

OUTPUT_DIR = Path("./content/smart_and_common_split_FINAL_PCA_trainonly_ANALYTIC_N3767")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR_CANDIDATES = [Path("./data"), Path("../data"), Path("../../data")]
DATA_DIR = None
for p in DATA_DIR_CANDIDATES:
    if (p / "lista_global_vars.csv").exists() and (p / "target_col.csv").exists():
        DATA_DIR = p
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find lista_global_vars.csv and target_col.csv")

FEATURES_FILE = DATA_DIR / "lista_global_vars.csv"
TARGET_FILE = DATA_DIR / "target_col.csv"

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore")

print("DATA_DIR:", DATA_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


DATA_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/data
OUTPUT_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/content/smart_and_common_split_FINAL_PCA_trainonly_ANALYTIC_N3767


In [2]:
# ============================================================
# 1. IMPORTS
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    average_precision_score,
    roc_auc_score,
)
import joblib

TF_AVAILABLE = False
if USE_TENSORFLOW_DNN:
    try:
        import tensorflow as tf
        from tensorflow.keras import layers, models
        from tensorflow.keras.callbacks import EarlyStopping
        TF_AVAILABLE = True

        tf.keras.backend.clear_session()
        tf.random.set_seed(RANDOM_STATE)
        try:
            tf.config.experimental.enable_op_determinism()
        except Exception:
            pass

        print("TensorFlow available:", tf.__version__)
    except Exception as e:
        print("TensorFlow not available. Falling back to weighted logistic regression for perpetration.")
        print("Detail:", e)
        TF_AVAILABLE = False
else:
    print("USE_TENSORFLOW_DNN=False. Using weighted logistic regression for perpetration.")

print("Imports OK")


TensorFlow available: 2.21.0
Imports OK


In [3]:
# ============================================================
# 2. HELPERS
# ============================================================

V_COL = "V.SUM.TOTAL"
P_COL = "P.SUM.TOTAL"

LEAKAGE_COLS = [
    "VÍCTIMA", "PERPETRADOR", "VICTIMA_PERPETRADOR",
    "POLIVICTIMIZACION", "POLIPERPETRACION",
    "SOLO.VICTIMA", "SOLO.PERPETRADOR", "NO.VICT_NO.PERP",
    "V.O", "P.SUM.TOTAL", "V.SUM.TOTAL",
    "INTERSECT"
]

SUBGROUP_COLS = [
    "PAÍS", "ETNIA.BN", "EDAD",
    "GENERO_BIN_0", "GENERO_BIN_1", "GENERO_BIN_2",
    "ORIENTSEX.BN_1", "ORIENTSEX.BN_2", "ORIENTSEX.BN_3",
]


def compute_metrics(y_true, y_pred, y_prob=None, label=None):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2 * ppv * recall / (ppv + recall) if (ppv + recall) > 0 else np.nan

    row = {
        "strategy": label,
        "n": int(len(y_true)),
        "positives": int(y_true.sum()),
        "negatives": int((y_true == 0).sum()),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "recall_sensitivity": recall,
        "specificity": specificity,
        "precision_ppv": ppv,
        "npv": npv,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_positive": f1,
        "accuracy": accuracy_score(y_true, y_pred),
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
    }

    if y_prob is not None:
        try:
            row["auc_pr"] = average_precision_score(y_true, y_prob)
        except Exception:
            row["auc_pr"] = np.nan
        try:
            row["auc_roc"] = roc_auc_score(y_true, y_prob)
        except Exception:
            row["auc_roc"] = np.nan

    return row


def pct_table(df, metric_cols=None):
    if metric_cols is None:
        metric_cols = [
            "recall_sensitivity", "specificity", "precision_ppv", "npv",
            "balanced_accuracy", "f1_positive", "accuracy", "fpr", "fnr",
            "auc_pr", "auc_roc"
        ]
    out = df.copy()
    for c in metric_cols:
        if c in out.columns:
            out[c] = out[c].apply(lambda x: np.nan if pd.isna(x) else round(x * 100, 1))
    return out


def make_sample_weights(y, pos_multiplier=1.0):
    y = np.asarray(y).astype(int)
    pos_rate = y.mean()
    neg_rate = 1 - pos_rate
    base_pos_weight = neg_rate / pos_rate if pos_rate > 0 else 1.0
    weights = np.where(y == 1, base_pos_weight * pos_multiplier, 1.0)
    return weights, float(base_pos_weight * pos_multiplier)


def fit_pca_train_only(X, train_idx, test_idx, prefix, output_dir=OUTPUT_DIR, pca_threshold=PCA_VARIANCE_THRESHOLD):
    X_train = X.loc[train_idx].copy()
    X_test = X.loc[test_idx].copy()

    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train.values)
    X_test_scaled = scaler.transform(X_test.values)

    means = X_train_scaled.mean(axis=0)
    X_train_centered = X_train_scaled - means
    X_test_centered = X_test_scaled - means

    pca_full = PCA(n_components=X_train.shape[1], random_state=RANDOM_STATE)
    X_train_pca_full = pca_full.fit_transform(X_train_centered)
    X_test_pca_full = pca_full.transform(X_test_centered)

    cum_var = np.cumsum(pca_full.explained_variance_ratio_)
    n_components = int(np.searchsorted(cum_var, pca_threshold, side="left") + 1)

    pc_cols = [f"PC{i+1}" for i in range(n_components)]
    X_train_pca = pd.DataFrame(X_train_pca_full[:, :n_components], index=train_idx, columns=pc_cols)
    X_test_pca = pd.DataFrame(X_test_pca_full[:, :n_components], index=test_idx, columns=pc_cols)

    pca_dir = output_dir / "pca_artifacts" / prefix
    pca_dir.mkdir(parents=True, exist_ok=True)

    joblib.dump(scaler, pca_dir / "scaler_minmax.joblib")
    joblib.dump(pca_full, pca_dir / "pca_full.joblib")
    pd.Series(means, index=X.columns, name="scaled_train_mean").to_csv(pca_dir / "scaled_train_means.csv")
    pd.DataFrame({"feature": list(X.columns)}).to_csv(pca_dir / "pca_input_feature_names.csv", index=False)

    variance_df = pd.DataFrame({
        "component": [f"PC{i+1}" for i in range(len(cum_var))],
        "component_number": np.arange(1, len(cum_var) + 1),
        "explained_variance_ratio": pca_full.explained_variance_ratio_,
        "explained_variance_percent": pca_full.explained_variance_ratio_ * 100,
        "cumulative_variance_ratio": cum_var,
        "cumulative_variance_percent": cum_var * 100,
        "retained_for_model": np.arange(1, len(cum_var) + 1) <= n_components,
    })
    variance_df.to_csv(pca_dir / "pca_variance.csv", index=False)

    meta = {
        "prefix": prefix,
        "pca_threshold": float(pca_threshold),
        "n_input_features": int(X_train.shape[1]),
        "n_components_retained": int(n_components),
        "retained_variance": float(cum_var[n_components - 1]),
        "pca_dir": str(pca_dir),
    }

    with open(pca_dir / "pca_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    return X_train_pca, X_test_pca, meta


def train_pruned_tree(X_train, y_train, X_test, y_test, threshold=VICTIM_THRESHOLD_DEFAULT):
    full_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
    full_tree.fit(X_train, y_train)

    path = full_tree.cost_complexity_pruning_path(X_train, y_train)
    ccp_alphas = np.unique(path.ccp_alphas)

    rows = []
    models_by_alpha = {}

    for alpha in ccp_alphas:
        clf = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=float(alpha))
        clf.fit(X_train, y_train)
        prob = clf.predict_proba(X_test)[:, 1]
        pred = (prob >= threshold).astype(int)

        m = compute_metrics(y_test, pred, prob, label="victim_tree_candidate")
        m["ccp_alpha"] = float(alpha)
        m["depth"] = int(clf.get_depth())
        m["n_leaves"] = int(clf.get_n_leaves())
        rows.append(m)
        models_by_alpha[float(alpha)] = clf

    alpha_df = pd.DataFrame(rows)

    candidates = alpha_df[alpha_df["recall_sensitivity"] >= VICTIM_MIN_RECALL_FOR_TREE_SELECTION].copy()

    if len(candidates) == 0:
        selected = alpha_df.sort_values(
            ["recall_sensitivity", "balanced_accuracy", "specificity"],
            ascending=False
        ).iloc[0]
        selection_note = f"No candidate reached recall >= {VICTIM_MIN_RECALL_FOR_TREE_SELECTION}; selected highest recall/balance."
    else:
        selected = candidates.sort_values(
            ["balanced_accuracy", "specificity", "precision_ppv", "n_leaves"],
            ascending=[False, False, False, True]
        ).iloc[0]
        selection_note = f"Selected among candidates with recall >= {VICTIM_MIN_RECALL_FOR_TREE_SELECTION}."

    best_alpha = float(selected["ccp_alpha"])
    model = models_by_alpha[best_alpha]

    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= threshold).astype(int)

    return model, prob, pred, alpha_df, selected.to_dict(), selection_note


def build_perp_dnn(input_dim):
    tf.keras.backend.clear_session()
    tf.random.set_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    random.seed(RANDOM_STATE)

    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(16, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        ],
    )
    return model


def train_perp_model(X_train, y_train, X_test, y_test, threshold=PERP_THRESHOLD_DEFAULT):
    weights, pos_weight = make_sample_weights(y_train, pos_multiplier=PERP_POS_WEIGHT_MULTIPLIER)

    if TF_AVAILABLE:
        model = build_perp_dnn(X_train.shape[1])
        callbacks = [
            EarlyStopping(monitor="val_recall", mode="max", patience=NN_PATIENCE, restore_best_weights=True, verbose=0)
        ]

        history = model.fit(
            X_train.values,
            np.asarray(y_train).astype(int),
            sample_weight=weights,
            validation_split=NN_VALIDATION_SPLIT,
            epochs=NN_EPOCHS,
            batch_size=NN_BATCH_SIZE,
            callbacks=callbacks,
            verbose=0,
            shuffle=True,
        )

        prob = model.predict(X_test.values, verbose=0).ravel()
        pred = (prob >= threshold).astype(int)
        return model, prob, pred, {
            "model_type": "DNN",
            "positive_weight": pos_weight,
            "epochs_ran": len(history.history.get("loss", [])),
            "history": {k: [float(vv) for vv in v] for k, v in history.history.items()},
        }

    model = LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE)
    model.fit(X_train.values, np.asarray(y_train).astype(int), sample_weight=weights)
    prob = model.predict_proba(X_test.values)[:, 1]
    pred = (prob >= threshold).astype(int)

    return model, prob, pred, {
        "model_type": "LogisticRegression fallback because TensorFlow was unavailable",
        "positive_weight": pos_weight,
    }


def train_overlap_logreg(X_train, y_train, X_test, y_test, threshold=OVERLAP_DIRECT_THRESHOLD):
    weights, pos_weight = make_sample_weights(y_train, pos_multiplier=OVERLAP_POS_WEIGHT_MULTIPLIER)

    model = LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE)
    model.fit(X_train.values, np.asarray(y_train).astype(int), sample_weight=weights)

    prob = model.predict_proba(X_test.values)[:, 1]
    pred = (prob >= threshold).astype(int)

    return model, prob, pred, {
        "model_type": "LogisticRegression_SW_pos1.5",
        "positive_weight": pos_weight,
    }


print("Helpers ready")


Helpers ready


In [4]:
# ============================================================
# 3. DATA PREPARATION
# ============================================================

feat_df = pd.read_csv(FEATURES_FILE)
target_df = pd.read_csv(TARGET_FILE).fillna(0)

print("Original features:", feat_df.shape)
print("Original targets:", target_df.shape)

df = feat_df.join(target_df, how="inner")
# Preserve raw row number for traceability; the analysis index is reset after filtering.
df["raw_row_index_before_filter"] = df.index.astype(int)

# Same analytic filtering used in the final overlap notebook:
# remove extremely sparse categories and drop their dummy columns.
required_filter_cols = ["GENERO_BIN_2", "ORIENTSEX.BN_3"]
missing_filter_cols = [c for c in required_filter_cols if c not in df.columns]
if missing_filter_cols:
    raise ValueError(f"Missing expected sparse-category columns: {missing_filter_cols}")

filter_mask = ~((df["GENERO_BIN_2"] == 1) | (df["ORIENTSEX.BN_3"] == 1))
df = (
    df.loc[filter_mask]
    .drop(columns=["GENERO_BIN_2", "ORIENTSEX.BN_3"])
    .reset_index(drop=True)
)

# Count-based outcomes. Do not use old VICTIMA_PERPETRADOR flag.
df["y_victim"] = (pd.to_numeric(df[V_COL], errors="coerce") >= 1).astype(int)
df["y_perp"] = (pd.to_numeric(df[P_COL], errors="coerce") >= 1).astype(int)
df["y_overlap"] = ((df["y_victim"] == 1) & (df["y_perp"] == 1)).astype(int)

# Feature matrix: remove target/leakage columns.
X = df.drop(columns=[c for c in LEAKAGE_COLS + ["y_victim", "y_perp", "y_overlap", "raw_row_index_before_filter"] if c in df.columns]).copy()

# Feature engineering consistent with final overlap notebook.
pd.set_option("future.no_silent_downcasting", True)

if "PAÍS" in X.columns:
    X["PAÍS"] = X["PAÍS"].replace({1: True, 2: False})
if "ETNIA.BN" in X.columns:
    X["ETNIA.BN"] = X["ETNIA.BN"].replace({0.0: False, 1.0: True})
if "FUGAS.BN" in X.columns:
    X["FUGAS.BN"] = X["FUGAS.BN"].replace({0.0: False, 1.0: True})

if {"GENERO_BIN_0", "GENERO_BIN_1"}.issubset(X.columns):
    X["GENERO.BN0"] = X["GENERO_BIN_0"].replace({0.0: False, 1.0: True})
    X["GENERO.BN1"] = X["GENERO_BIN_1"].replace({0.0: False, 1.0: True})
    X = X.drop(columns=["GENERO_BIN_0", "GENERO_BIN_1"])

if {"ORIENTSEX.BN_1", "ORIENTSEX.BN_2"}.issubset(X.columns):
    X["ORIENTSEX.BN0"] = X["ORIENTSEX.BN_1"].replace({0.0: False, 1.0: True})
    X["ORIENTSEX.BN1"] = X["ORIENTSEX.BN_2"].replace({0.0: False, 1.0: True})
    X = X.drop(columns=["ORIENTSEX.BN_1", "ORIENTSEX.BN_2"])

if "CONVIVEN.5" in X.columns:
    X = X.rename(columns={"CONVIVEN.5": "CONVIVEN_H"})
    X["CONVIVEN_H"] = X["CONVIVEN_H"].replace({0.0: False, 1.0: True})

if "CONVIVEN.6" in X.columns:
    X = X.rename(columns={"CONVIVEN.6": "CONVIVEN_0"})
    X["CONVIVEN_0"] = X["CONVIVEN_0"].replace({0.0: False, 1.0: True})

bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)
X = X.apply(pd.to_numeric, errors="raise")

# Metadata for row-level outputs.
meta_cols = [V_COL, P_COL, "y_victim", "y_perp", "y_overlap", "raw_row_index_before_filter"]
meta_cols += [c for c in SUBGROUP_COLS if c in df.columns]
df_meta = df[meta_cols].copy()
df_meta["idx_original"] = df_meta.index.astype(int)

y_victim = df["y_victim"].astype(int)
y_perp = df["y_perp"].astype(int)
y_overlap = df["y_overlap"].astype(int)

print("\nAnalytical sample n:", len(X))
print("Feature matrix:", X.shape)
print("Feature columns:", list(X.columns))
print("Missing feature values:", int(X.isna().sum().sum()))

print("\nOutcome prevalences:")
for name, y in [("victimization", y_victim), ("perpetration", y_perp), ("overlap", y_overlap)]:
    print(name, int(y.sum()), "/", len(y), f"= {100*y.mean():.1f}%")


# Hard validation: this benchmark must use the final analytical sample, not the raw n=4,024 sample.
if len(X) != EXPECTED_ANALYTIC_N:
    raise RuntimeError(
        f"Analytical sample mismatch: got n={len(X)}, expected n={EXPECTED_ANALYTIC_N}. "
        "Do not use this Smart AND benchmark until the filtering matches the final paper sample."
    )

print("\nFour-category role profile:")
role_table = pd.crosstab(y_victim, y_perp, rownames=["victim"], colnames=["perpetrator"], margins=True)
display(role_table)

X.to_csv(OUTPUT_DIR / "X_features_clean.csv", index=True)
pd.DataFrame({"y_victim": y_victim, "y_perp": y_perp, "y_overlap": y_overlap}).to_csv(OUTPUT_DIR / "targets_count_based.csv", index=True)
df_meta.to_csv(OUTPUT_DIR / "metadata_count_based.csv", index=False)


Original features: (4024, 29)
Original targets: (4024, 11)

Analytical sample n: 3767
Feature matrix: (3767, 27)
Feature columns: ['PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2', 'CONVIVEN.3', 'CONVIVEN.4', 'CONVIVEN_H', 'CONVIVEN_0', 'AUTOEFIC.MEAN', 'AUTOEFIC.VAR', 'IMPULS.MEAN', 'IMPULS.MEDIAN', 'IMPULS.VAR', 'APOYO.MEAN', 'APOYO.MEDIAN', 'APOYO.VAR', 'MORAL.MEAN', 'MORAL.VAR', 'PORNO.T', 'GENERO.BN0', 'GENERO.BN1', 'ORIENTSEX.BN0', 'ORIENTSEX.BN1']
Missing feature values: 0

Outcome prevalences:
victimization 1861 / 3767 = 49.4%
perpetration 885 / 3767 = 23.5%
overlap 713 / 3767 = 18.9%

Four-category role profile:


perpetrator,0,1,All
victim,,,
0,1734,172,1906
1,1148,713,1861
All,2882,885,3767


In [5]:
# ============================================================
# 4. COMMON SPLIT STRATIFIED BY FOUR-CATEGORY ROLE PROFILE
# ============================================================

idx_all = X.index

joint_strata = y_victim.astype(str) + "_" + y_perp.astype(str)

idx_train, idx_test = train_test_split(
    idx_all,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=joint_strata,
)

idx_train = pd.Index(idx_train)
idx_test = pd.Index(idx_test)


# Hard validation: common split must match final held-out test size.
if len(idx_test) != EXPECTED_TEST_N:
    raise RuntimeError(
        f"Common test size mismatch: got n={len(idx_test)}, expected n={EXPECTED_TEST_N}. "
        "This usually means the raw n=4,024 sample was used instead of analytical n=3,767."
    )

if len(idx_train) + len(idx_test) != EXPECTED_ANALYTIC_N:
    raise RuntimeError(
        f"Split total mismatch: train+test={len(idx_train)+len(idx_test)}, expected {EXPECTED_ANALYTIC_N}."
    )

print("Train n:", len(idx_train))
print("Test n:", len(idx_test))

print("\nTest outcome counts:")
print("victim positives:", int(y_victim.loc[idx_test].sum()))
print("perp positives:", int(y_perp.loc[idx_test].sum()))
print("overlap positives:", int(y_overlap.loc[idx_test].sum()))

print("\nTrain role table:")
display(pd.crosstab(y_victim.loc[idx_train], y_perp.loc[idx_train], rownames=["victim"], colnames=["perpetrator"], margins=True))

print("\nTest role table:")
display(pd.crosstab(y_victim.loc[idx_test], y_perp.loc[idx_test], rownames=["victim"], colnames=["perpetrator"], margins=True))

split_info = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "stratification": "joint four-category profile: y_victim + y_perp",
    "train_idx": list(map(int, idx_train)),
    "test_idx": list(map(int, idx_test)),
    "n_train": int(len(idx_train)),
    "n_test": int(len(idx_test)),
}

with open(OUTPUT_DIR / "common_split_indices.json", "w", encoding="utf-8") as f:
    json.dump(split_info, f, indent=2)

pd.DataFrame({"idx_train": idx_train}).to_csv(OUTPUT_DIR / "common_train_indices.csv", index=False)
pd.DataFrame({"idx_test": idx_test}).to_csv(OUTPUT_DIR / "common_test_indices.csv", index=False)


Train n: 2825
Test n: 942

Test outcome counts:
victim positives: 465
perp positives: 221
overlap positives: 178

Train role table:


perpetrator,0,1,All
victim,,,
0,1300,129,1429
1,861,535,1396
All,2161,664,2825



Test role table:


perpetrator,0,1,All
victim,,,
0,434,43,477
1,287,178,465
All,721,221,942


In [6]:
# ============================================================
# 5. TRAIN MODELS UNDER COMMON SPLIT
# ============================================================

# Victimization model: pruned decision tree, PCA train-only.
Xv_train_pca, Xv_test_pca, victim_pca_meta = fit_pca_train_only(X, idx_train, idx_test, prefix="victimization")
yv_train = y_victim.loc[idx_train]
yv_test = y_victim.loc[idx_test]

victim_model, victim_prob, victim_pred, victim_alpha_df, victim_selected, victim_selection_note = train_pruned_tree(
    Xv_train_pca, yv_train, Xv_test_pca, yv_test, threshold=VICTIM_THRESHOLD_DEFAULT
)

victim_alpha_df.to_csv(OUTPUT_DIR / "victim_tree_alpha_candidates.csv", index=False)

print("Victimization model")
print("PCA:", victim_pca_meta)
print("Selected tree:", victim_selected)
print(victim_selection_note)
print(classification_report(yv_test, victim_pred, digits=3, zero_division=0))


# Perpetration model: final-style DNN if TensorFlow available; otherwise weighted logistic fallback.
Xp_train_pca, Xp_test_pca, perp_pca_meta = fit_pca_train_only(X, idx_train, idx_test, prefix="perpetration")
yp_train = y_perp.loc[idx_train]
yp_test = y_perp.loc[idx_test]

perp_model, perp_prob, perp_pred, perp_meta = train_perp_model(
    Xp_train_pca, yp_train, Xp_test_pca, yp_test, threshold=PERP_THRESHOLD_DEFAULT
)

print("\nPerpetration model")
print("PCA:", perp_pca_meta)
print("Model meta:", {k: v for k, v in perp_meta.items() if k != "history"})
print(classification_report(yp_test, perp_pred, digits=3, zero_division=0))


# Direct overlap model: final-style LogisticRegression_SW_pos1.5, PCA train-only.
Xo_train_pca, Xo_test_pca, overlap_pca_meta = fit_pca_train_only(X, idx_train, idx_test, prefix="direct_overlap")
yo_train = y_overlap.loc[idx_train]
yo_test = y_overlap.loc[idx_test]

overlap_model, overlap_prob, overlap_pred, overlap_meta = train_overlap_logreg(
    Xo_train_pca, yo_train, Xo_test_pca, yo_test, threshold=OVERLAP_DIRECT_THRESHOLD
)

print("\nDirect overlap model")
print("PCA:", overlap_pca_meta)
print("Model meta:", overlap_meta)
print(classification_report(yo_test, overlap_pred, digits=3, zero_division=0))


# Save models
model_dir = OUTPUT_DIR / "models"
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(victim_model, model_dir / "victim_pruned_tree_common_split.joblib")
joblib.dump(overlap_model, model_dir / "direct_overlap_logreg_SW_pos1p5_common_split.joblib")

if TF_AVAILABLE and perp_meta.get("model_type") == "DNN":
    perp_model.save(model_dir / "perp_dnn_common_split.keras")
else:
    joblib.dump(perp_model, model_dir / "perp_model_common_split.joblib")


Victimization model
PCA: {'prefix': 'victimization', 'pca_threshold': 0.95, 'n_input_features': 27, 'n_components_retained': 18, 'retained_variance': 0.9505009259538393, 'pca_dir': 'content/smart_and_common_split_FINAL_PCA_trainonly_ANALYTIC_N3767/pca_artifacts/victimization'}
Selected tree: {'strategy': 'victim_tree_candidate', 'n': 942, 'positives': 465, 'negatives': 477, 'TP': 347, 'FP': 287, 'TN': 190, 'FN': 118, 'recall_sensitivity': 0.7462365591397849, 'specificity': 0.39832285115303984, 'precision_ppv': 0.5473186119873817, 'npv': 0.6168831168831169, 'balanced_accuracy': 0.5722797051464124, 'f1_positive': 0.6314831665150137, 'accuracy': 0.5700636942675159, 'fpr': 0.6016771488469602, 'fnr': 0.2537634408602151, 'auc_pr': 0.5764017393405033, 'auc_roc': 0.6116160591510561, 'ccp_alpha': 0.010415004024879937, 'depth': 2, 'n_leaves': 3}
No candidate reached recall >= 0.85; selected highest recall/balance.
              precision    recall  f1-score   support

           0      0.617    

E0000 00:00:1783659896.697560 13805873 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}



Perpetration model
PCA: {'prefix': 'perpetration', 'pca_threshold': 0.95, 'n_input_features': 27, 'n_components_retained': 18, 'retained_variance': 0.9505009259538393, 'pca_dir': 'content/smart_and_common_split_FINAL_PCA_trainonly_ANALYTIC_N3767/pca_artifacts/perpetration'}
Model meta: {'model_type': 'DNN', 'positive_weight': 3.2545180722891565, 'epochs_ran': 45}
              precision    recall  f1-score   support

           0      0.852     0.656     0.741       721
           1      0.359     0.629     0.457       221

    accuracy                          0.650       942
   macro avg      0.606     0.642     0.599       942
weighted avg      0.737     0.650     0.675       942


Direct overlap model
PCA: {'prefix': 'direct_overlap', 'pca_threshold': 0.95, 'n_input_features': 27, 'n_components_retained': 18, 'retained_variance': 0.9505009259538393, 'pca_dir': 'content/smart_and_common_split_FINAL_PCA_trainonly_ANALYTIC_N3767/pca_artifacts/direct_overlap'}
Model meta: {'model_type

In [7]:
# ============================================================
# 6. DIRECT OVERLAP VS SMART AND BENCHMARK
# ============================================================

# Smart AND default strategy: same common test set, role-specific predictions.
smart_and_default_pred = ((victim_pred == 1) & (perp_pred == 1)).astype(int)
smart_and_default_prob = np.minimum(victim_prob, perp_prob)

direct_metrics = compute_metrics(
    yo_test,
    overlap_pred,
    overlap_prob,
    label="Direct overlap classifier — LogisticRegression_SW_pos1.5, threshold=0.50"
)

smart_default_metrics = compute_metrics(
    yo_test,
    smart_and_default_pred,
    smart_and_default_prob,
    label=f"Smart AND default — victim≥{VICTIM_THRESHOLD_DEFAULT:.2f} AND perp≥{PERP_THRESHOLD_DEFAULT:.2f}"
)

# Grid search over role thresholds for Smart AND.
thresholds = np.round(np.arange(GRID_MIN, GRID_MAX + 1e-9, GRID_STEP), 3)

grid_rows = []
for vt in thresholds:
    v_pred_t = (victim_prob >= vt).astype(int)
    for pt in thresholds:
        p_pred_t = (perp_prob >= pt).astype(int)
        smart_pred = ((v_pred_t == 1) & (p_pred_t == 1)).astype(int)
        smart_prob = np.minimum(victim_prob, perp_prob)
        m = compute_metrics(yo_test, smart_pred, smart_prob, label="Smart AND grid")
        m["victim_threshold"] = float(vt)
        m["perp_threshold"] = float(pt)
        grid_rows.append(m)

smart_grid = pd.DataFrame(grid_rows)

direct_recall = direct_metrics["recall_sensitivity"]

# Recall-matched benchmark: among Smart AND pairs with recall >= direct recall,
# choose the pair with highest balanced accuracy, then specificity, PPV.
candidates = smart_grid[smart_grid["recall_sensitivity"] >= direct_recall].copy()

if len(candidates) > 0:
    smart_matched = (
        candidates
        .sort_values(["balanced_accuracy", "specificity", "precision_ppv"], ascending=False)
        .iloc[0]
        .to_dict()
    )
    matched_note = "Smart AND threshold pair selected among candidates with recall >= direct-overlap recall."
else:
    tmp = smart_grid.copy()
    tmp["recall_gap_abs"] = (tmp["recall_sensitivity"] - direct_recall).abs()
    smart_matched = (
        tmp
        .sort_values(["recall_gap_abs", "balanced_accuracy", "specificity"], ascending=[True, False, False])
        .iloc[0]
        .to_dict()
    )
    matched_note = "No Smart AND pair reached direct-overlap recall; selected closest recall."

smart_matched["strategy"] = (
    f"Smart AND recall-matched — victim≥{smart_matched['victim_threshold']:.2f} "
    f"AND perp≥{smart_matched['perp_threshold']:.2f}"
)

comparison_df = pd.DataFrame([direct_metrics, smart_default_metrics, smart_matched])
comparison_df["benchmark_note"] = [
    "Direct overlap model trained on the dual-role target under the common split.",
    "Conjunctive rule using role-specific default thresholds.",
    matched_note,
]

first_cols = [
    "strategy", "victim_threshold", "perp_threshold",
    "n", "positives", "negatives",
    "TP", "FP", "TN", "FN",
    "recall_sensitivity", "specificity", "precision_ppv", "npv",
    "balanced_accuracy", "f1_positive", "accuracy",
    "fpr", "fnr", "auc_pr", "auc_roc",
    "benchmark_note",
]
comparison_df = comparison_df[[c for c in first_cols if c in comparison_df.columns] + [c for c in comparison_df.columns if c not in first_cols]]
comparison_pretty = pct_table(comparison_df)

smart_grid.to_csv(OUTPUT_DIR / "smart_and_threshold_grid_raw.csv", index=False)
pct_table(smart_grid).to_csv(OUTPUT_DIR / "smart_and_threshold_grid_percent.csv", index=False)

comparison_df.to_csv(OUTPUT_DIR / "strategy_benchmark_common_split_raw.csv", index=False)
comparison_pretty.to_csv(OUTPUT_DIR / "strategy_benchmark_common_split_percent.csv", index=False)

print("=== STRATEGY BENCHMARK — COMMON SPLIT RAW ===")
display(comparison_df)

print("=== STRATEGY BENCHMARK — COMMON SPLIT PERCENT ===")
display(comparison_pretty)


=== STRATEGY BENCHMARK — COMMON SPLIT RAW ===


,strategy,victim_threshold,perp_threshold,n,positives,negatives,TP,FP,TN,FN,...,precision_ppv,npv,balanced_accuracy,f1_positive,accuracy,fpr,fnr,auc_pr,auc_roc,benchmark_note
0,Direct overlap classifier — LogisticRegression...,NaN,NaN,942,178,764,139,337,427,39,...,0.292017,0.916309,0.669900,0.425076,0.600849,0.441099,0.219101,0.403045,0.747676,Direct overlap model trained on the dual-role ...
1,Smart AND default — victim≥0.50 AND perp≥0.50,NaN,NaN,942,178,764,106,225,539,72,...,0.320242,0.882160,0.650502,0.416503,0.684713,0.294503,0.404494,0.399607,0.721329,Conjunctive rule using role-specific default t...
2,Smart AND recall-matched — victim≥0.01 AND per...,0.01,0.43,942,178,764,142,357,407,36,...,0.284569,0.918736,0.665238,0.419498,0.582803,0.467277,0.202247,0.399607,0.721329,Smart AND threshold pair selected among candid...


=== STRATEGY BENCHMARK — COMMON SPLIT PERCENT ===


,strategy,victim_threshold,perp_threshold,n,positives,negatives,TP,FP,TN,FN,...,precision_ppv,npv,balanced_accuracy,f1_positive,accuracy,fpr,fnr,auc_pr,auc_roc,benchmark_note
0,Direct overlap classifier — LogisticRegression...,NaN,NaN,942,178,764,139,337,427,39,...,29.2,91.6,67.0,42.5,60.1,44.1,21.9,40.3,74.8,Direct overlap model trained on the dual-role ...
1,Smart AND default — victim≥0.50 AND perp≥0.50,NaN,NaN,942,178,764,106,225,539,72,...,32.0,88.2,65.1,41.7,68.5,29.5,40.4,40.0,72.1,Conjunctive rule using role-specific default t...
2,Smart AND recall-matched — victim≥0.01 AND per...,0.01,0.43,942,178,764,142,357,407,36,...,28.5,91.9,66.5,41.9,58.3,46.7,20.2,40.0,72.1,Smart AND threshold pair selected among candid...


In [8]:
# ============================================================
# 7. SAVE ROW-LEVEL PREDICTIONS
# ============================================================

test_meta = df_meta.loc[idx_test].copy()

predictions_common = pd.DataFrame({
    "idx_original": idx_test.astype(int),
    "y_true_victim": yv_test.values.astype(int),
    "y_prob_victim": victim_prob.astype(float),
    "y_pred_victim_default": victim_pred.astype(int),

    "y_true_perp": yp_test.values.astype(int),
    "y_prob_perp": perp_prob.astype(float),
    "y_pred_perp_default": perp_pred.astype(int),

    "y_true_overlap": yo_test.values.astype(int),
    "y_prob_overlap_direct": overlap_prob.astype(float),
    "y_pred_overlap_direct": overlap_pred.astype(int),

    "y_prob_smart_and_min": smart_and_default_prob.astype(float),
    "y_pred_smart_and_default": smart_and_default_pred.astype(int),
})

# Add recall-matched Smart AND predictions
best_vt = float(smart_matched["victim_threshold"])
best_pt = float(smart_matched["perp_threshold"])

predictions_common["y_pred_smart_and_recall_matched"] = (
    ((predictions_common["y_prob_victim"] >= best_vt) & (predictions_common["y_prob_perp"] >= best_pt))
    .astype(int)
)
predictions_common["smart_and_recall_matched_victim_threshold"] = best_vt
predictions_common["smart_and_recall_matched_perp_threshold"] = best_pt

# Add counts/subgroups for traceability
test_meta_reset = test_meta.reset_index(drop=True)
for col in test_meta_reset.columns:
    if col not in predictions_common.columns:
        predictions_common[col] = test_meta_reset[col].values

# Validate official overlap in row-level table
validation_match = (
    predictions_common["y_true_overlap"].astype(int).to_numpy()
    ==
    (((predictions_common[V_COL] >= 1) & (predictions_common[P_COL] >= 1)).astype(int).to_numpy())
).mean()

print("validation y_true_overlap vs count-based overlap:", validation_match)
if validation_match < 0.999:
    raise RuntimeError("Row-level overlap validation failed.")

predictions_common.to_csv(OUTPUT_DIR / "predictions_common_split_all_strategies.csv", index=False)

print("Saved row-level predictions:")
print(OUTPUT_DIR / "predictions_common_split_all_strategies.csv")
print("Shape:", predictions_common.shape)
display(predictions_common.head())


validation y_true_overlap vs count-based overlap: 1.0
Saved row-level predictions:
content/smart_and_common_split_FINAL_PCA_trainonly_ANALYTIC_N3767/predictions_common_split_all_strategies.csv
Shape: (942, 28)


,idx_original,y_true_victim,y_prob_victim,y_pred_victim_default,y_true_perp,y_prob_perp,y_pred_perp_default,y_true_overlap,y_prob_overlap_direct,y_pred_overlap_direct,...,y_perp,y_overlap,raw_row_index_before_filter,PAÍS,ETNIA.BN,EDAD,GENERO_BIN_0,GENERO_BIN_1,ORIENTSEX.BN_1,ORIENTSEX.BN_2
0,2677,0,0.521379,1,0,0.697807,1,0,0.767175,1,...,0,0,2855,1,1.0,15.0,1,0,1,0
1,3689,0,0.327273,0,0,0.279404,0,0,0.309353,0,...,0,0,3945,1,0.0,14.0,1,0,1,0
2,478,0,0.327273,0,0,0.411657,0,0,0.402212,0,...,0,0,501,1,0.0,16.0,1,0,1,0
3,42,0,0.521379,1,0,0.526374,1,0,0.554259,1,...,0,0,43,2,1.0,14.0,1,0,1,0
4,2989,0,0.759091,1,0,0.690039,1,0,0.725642,1,...,0,0,3185,1,0.0,16.0,1,0,1,0


In [9]:
# ============================================================
# 8. MANUSCRIPT-READY TABLE + CONFIG
# ============================================================

table = comparison_df.copy()

# Keep the two rows most likely to be reported:
# direct overlap and recall-matched Smart AND.
report_table = table[
    table["strategy"].str.contains("Direct overlap|recall-matched", case=False, regex=True)
].copy()

report_table_pretty = pct_table(report_table)

report_cols = [
    "strategy",
    "TP", "FP", "TN", "FN",
    "recall_sensitivity", "specificity", "precision_ppv", "npv",
    "balanced_accuracy", "f1_positive", "accuracy",
    "victim_threshold", "perp_threshold",
    "benchmark_note",
]
report_table = report_table[[c for c in report_cols if c in report_table.columns]]
report_table_pretty = report_table_pretty[[c for c in report_cols if c in report_table_pretty.columns]]

report_table.to_csv(OUTPUT_DIR / "table_S_smart_and_common_split_raw.csv", index=False)
report_table_pretty.to_csv(OUTPUT_DIR / "table_S_smart_and_common_split_percent.csv", index=False)

config = {
    "purpose": "Strategy-level benchmark: direct overlap classifier vs Smart AND conjunctive rule",
    "important_interpretation": (
        "This benchmark uses a common train/test split and should be interpreted as a comparison "
        "of detection strategies, not as the direct combination of the separately optimized final saved classifiers."
    ),
    "data_files": {
        "features_file": str(FEATURES_FILE),
        "target_file": str(TARGET_FILE),
    },
    "analytical_sample_n": int(len(X)),
    "test_n": int(len(idx_test)),
    "validation_expected_n": {"analytical_n": EXPECTED_ANALYTIC_N, "test_n": EXPECTED_TEST_N},
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "stratification": "joint four-category role profile: victimization x perpetration",
    "outcome_definitions": {
        "victimization": "V.SUM.TOTAL >= 1",
        "perpetration": "P.SUM.TOTAL >= 1",
        "overlap": "V.SUM.TOTAL >= 1 AND P.SUM.TOTAL >= 1",
    },
    "pca": {
        "scope": "train-only",
        "threshold": PCA_VARIANCE_THRESHOLD,
        "victimization": victim_pca_meta,
        "perpetration": perp_pca_meta,
        "direct_overlap": overlap_pca_meta,
    },
    "models": {
        "victimization": {
            "algorithm": "cost-complexity pruned decision tree",
            "threshold": VICTIM_THRESHOLD_DEFAULT,
            "selection_note": victim_selection_note,
            "selected_tree": victim_selected,
        },
        "perpetration": {
            "algorithm": perp_meta.get("model_type"),
            "threshold": PERP_THRESHOLD_DEFAULT,
            "meta": {k: v for k, v in perp_meta.items() if k != "history"},
        },
        "direct_overlap": {
            "algorithm": "LogisticRegression_SW_pos1.5",
            "threshold": OVERLAP_DIRECT_THRESHOLD,
            "meta": overlap_meta,
        },
        "smart_and": {
            "default_rule": f"victim >= {VICTIM_THRESHOLD_DEFAULT} AND perpetration >= {PERP_THRESHOLD_DEFAULT}",
            "recall_matched_rule": f"victim >= {best_vt} AND perpetration >= {best_pt}",
            "selection_note": matched_note,
        },
    },
    "outputs": {
        "strategy_benchmark_raw": str(OUTPUT_DIR / "strategy_benchmark_common_split_raw.csv"),
        "strategy_benchmark_percent": str(OUTPUT_DIR / "strategy_benchmark_common_split_percent.csv"),
        "report_table_raw": str(OUTPUT_DIR / "table_S_smart_and_common_split_raw.csv"),
        "report_table_percent": str(OUTPUT_DIR / "table_S_smart_and_common_split_percent.csv"),
        "row_level_predictions": str(OUTPUT_DIR / "predictions_common_split_all_strategies.csv"),
        "threshold_grid": str(OUTPUT_DIR / "smart_and_threshold_grid_raw.csv"),
    },
}

with open(OUTPUT_DIR / "config_smart_and_common_split_FINAL.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

readme = f"""SMART AND common-split benchmark - FINAL clean version, analytical sample n=3,767.

This analysis is a strategy-level benchmark, not the direct combination of separately optimized final saved classifiers.

Common split:
- Analytical sample n = {len(X)}
- Held-out test n = {len(idx_test)}
- Stratified by joint four-category role profile: victimization x perpetration
- Random state = {RANDOM_STATE}

Outcome definitions:
- victimization = V.SUM.TOTAL >= 1
- perpetration = P.SUM.TOTAL >= 1
- overlap = V.SUM.TOTAL >= 1 AND P.SUM.TOTAL >= 1

Benchmark:
- Direct overlap classifier: LogisticRegression_SW_pos1.5 trained directly on overlap.
- Smart AND: adolescent classified as overlap only if both victimization and perpetration role-specific predictions are positive.
- Recall-matched Smart AND thresholds were selected on the same held-out benchmark predictions for descriptive comparison.

Do not report this as the direct combination of the final separately optimized production models.
"""

with open(OUTPUT_DIR / "README.txt", "w", encoding="utf-8") as f:
    f.write(readme)

print("=== MANUSCRIPT-READY SMART AND TABLE — RAW ===")
display(report_table)

print("=== MANUSCRIPT-READY SMART AND TABLE — PERCENT ===")
display(report_table_pretty)

print("\nSaved all outputs to:")
print(OUTPUT_DIR.resolve())


=== MANUSCRIPT-READY SMART AND TABLE — RAW ===


,strategy,TP,FP,TN,FN,recall_sensitivity,specificity,precision_ppv,npv,balanced_accuracy,f1_positive,accuracy,victim_threshold,perp_threshold,benchmark_note
0,Direct overlap classifier — LogisticRegression...,139,337,427,39,0.780899,0.558901,0.292017,0.916309,0.669900,0.425076,0.600849,NaN,NaN,Direct overlap model trained on the dual-role ...
2,Smart AND recall-matched — victim≥0.01 AND per...,142,357,407,36,0.797753,0.532723,0.284569,0.918736,0.665238,0.419498,0.582803,0.01,0.43,Smart AND threshold pair selected among candid...


=== MANUSCRIPT-READY SMART AND TABLE — PERCENT ===


,strategy,TP,FP,TN,FN,recall_sensitivity,specificity,precision_ppv,npv,balanced_accuracy,f1_positive,accuracy,victim_threshold,perp_threshold,benchmark_note
0,Direct overlap classifier — LogisticRegression...,139,337,427,39,78.1,55.9,29.2,91.6,67.0,42.5,60.1,NaN,NaN,Direct overlap model trained on the dual-role ...
2,Smart AND recall-matched — victim≥0.01 AND per...,142,357,407,36,79.8,53.3,28.5,91.9,66.5,41.9,58.3,0.01,0.43,Smart AND threshold pair selected among candid...



Saved all outputs to:
/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/content/smart_and_common_split_FINAL_PCA_trainonly_ANALYTIC_N3767


## Reporting note

Use this benchmark only if the direct overlap classifier clearly improves over the Smart AND strategy under the common split.

Recommended interpretation if retained:

> The Smart AND benchmark was evaluated as a strategy-level conjunctive rule under a common split. It should not be interpreted as the direct combination of the separately optimized final victimization and perpetration classifiers. The benchmark assesses whether victim-perpetrator overlap is better represented as a direct prediction target or as a derived conjunction of role-specific predictions.

If the recall-matched Smart AND strategy performs similarly or better than the direct overlap classifier, leave this analysis in internal checks or report it neutrally in the supplement.
